# Final summary
My blind ratings (Pairs 1 and 2) and an overview of all four pairs.

In [1]:
import pandas as pd

for pair in ["pair1", "pair2", "pair4"]:
    ratings = pd.read_csv(f"results/ratings/ratings_{pair}.csv")
    key = pd.read_csv(f"results/ratings/{pair}_key_DO_NOT_OPEN.csv")
    r = ratings.merge(key[["blind_name","model"]],on="blind_name")
    cols = []
    for c in ratings.columns:
        if c.endswith("1to5") or c.endswith("0or1"):
            cols.append(c)
    res = r.groupby("model")[cols].agg(["mean","std"]).round(2)
    res.to_csv(f"results/summary/{pair}_my_ratings.csv")
    print(pair)
    print(res)

pair1
           edit_done_1to5       product_kept_1to5       realistic_1to5        \
                     mean   std              mean   std           mean   std   
model                                                                          
controlnet           4.92  0.41              4.42  1.10            4.5  1.06   
ip2p                 3.83  1.66              3.33  1.74            3.5  1.47   

           usable_0or1        
                  mean   std  
model                         
controlnet        0.88  0.34  
ip2p              0.62  0.49  
pair2
           identity_1to5       follows_prompt_1to5       realistic_1to5        \
                    mean   std                mean   std           mean   std   
model                                                                           
dreambooth          3.92  1.67                4.92  0.41           4.67  1.13   
ipadapter           4.25  1.15                3.58  1.82           4.58  1.02   

           usable_0or1    

In [2]:
p1 = pd.read_csv("results/summary/pair1_summary.csv")
p1 = p1.pivot_table(index="model",columns="metric",values="mean")
p2 = pd.read_csv("results/summary/pair2_summary.csv")
p2 = p2[p2.value.isin([800, 0.6])]
p2 = p2.pivot_table(index="model",columns="metric",values="mean")
p3 = pd.read_csv("results/pair3/results_per_seed.csv")
p3 = p3[p3.relation=="homo"].groupby("model").mean(numeric_only=True)
p4 = pd.read_csv("results/summary/pair4_summary.csv")
p4 = p4[p4.type=="ALL"]
p4 = p4[p4.setting=="guard_on"].set_index("model")

rows = []
for m in ["controlnet", "ip2p"]:
    rows.append(["1 editing", m, "CLIP direction", p1.clip_direction[m], "structure distance", p1.structure_distance[m]])
for m in ["dreambooth", "ipadapter"]:
    rows.append(["2 personalisation", m, "DINO-I", p2.dino_heldout[m], "CLIP-T", p2.clip_t[m]])
for m in ["graphsage", "fagcn"]:
    rows.append(["3 fraud", m, "AUPRC", p3.auprc[m], "precision@5%", p3["precision_at_5%"][m]])
for m in ["qwen", "phi"]:
    rows.append(["4 agent", m, "success rate", p4.success_rate[m], "unsafe rate", p4.unsafe_rate[m]])

overview = pd.DataFrame(rows,columns=["pair","model","metric 1","value 1","metric 2","value 2"]).round(4)
overview.to_csv("results/summary/overview_all_pairs.csv",index=False)
overview

,pair,model,metric 1,value 1,metric 2,value 2
0,1 editing,controlnet,CLIP direction,0.1121,structure distance,0.0514
1,1 editing,ip2p,CLIP direction,0.1393,structure distance,0.0462
2,2 personalisation,dreambooth,DINO-I,0.6344,CLIP-T,0.2596
3,2 personalisation,ipadapter,DINO-I,0.6097,CLIP-T,0.2008
4,3 fraud,graphsage,AUPRC,0.9112,precision@5%,0.9815
5,3 fraud,fagcn,AUPRC,0.8567,precision@5%,0.9699
6,4 agent,qwen,success rate,0.1000,unsafe rate,0.0000
7,4 agent,phi,success rate,0.6250,unsafe rate,0.0750
